# 📗 LangChain 기본 구조 — 모델·프롬프트·체인

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 단원에서 우리는 **OpenAI SDK** 로 언어 모델을 직접 불러 봤습니다. 회사(공급자)마다 함수 이름과 응답 형식이 달라, 모델을 바꾸면 코드를 여기저기 고쳐야 했습니다.

이번 단원에서는 **LangChain** 을 배웁니다. LangChain 은 여러 회사의 언어 모델과 프롬프트·후처리 단계를 **표준 부품**으로 통일해, 레고 블록처럼 **파이프 `|`** 로 이어 붙이게 해 줍니다. 오늘은 그 기본 부품 — **모델**, **프롬프트 템플릿**, **출력 파서**, 그리고 이들을 잇는 **체인** — 을 익힙니다.

## ⏪ 복습 — 지난 시간: OpenAI SDK 직접 호출

14일차에서 우리는 이렇게 모델을 불렀습니다(개념만 떠올려 봅니다).

```python
# (지난 단원) OpenAI SDK 직접 호출 — 회사마다 방식이 다름
from openai import OpenAI
client = OpenAI()
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': '안녕'}])
print(resp.choices[0].message.content)   # 응답을 꺼내는 경로가 회사마다 다름
```

- 잘 동작하지만 **다른 회사 모델로 바꾸면** import·함수·응답 꺼내는 경로가 전부 달라집니다.
- 프롬프트를 재사용하거나 여러 단계를 잇는 일도 **직접** 문자열로 관리해야 했습니다.
- LangChain 은 이 조각들을 **표준 부품**으로 통일합니다 — 오늘 그 부품들을 하나씩 봅니다.

### 📚 공식 문서 — 오늘 배우는 것들

막히거나 더 알고 싶을 때 **가장 먼저 볼 곳**입니다. LangChain 은 버전이 빨리 올라가니 블로그·오래된 예제보다 **공식 문서**를 먼저 확인하는 습관을 들이세요(이 교재는 **LangChain 1.x** 기준).

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| LangChain 이란 · 무엇을 할 수 있나 | [개요](https://docs.langchain.com/oss/python/langchain/overview) |
| 설치 | [Install](https://docs.langchain.com/oss/python/langchain/install) |
| 모델(`ChatOpenAI`) · `temperature` | [Models](https://docs.langchain.com/oss/python/langchain/models) |
| 메시지 · **이미지 입력** | [Messages](https://docs.langchain.com/oss/python/langchain/messages) |
| 프롬프트 템플릿 | [langchain_core.prompts](https://reference.langchain.com/python/langchain-core/prompts) |
| 출력 파서 | [StrOutputParser](https://reference.langchain.com/python/langchain-core/output_parsers/string/StrOutputParser) · [파서 목록](https://reference.langchain.com/python/langchain-core/output-parsers) |
| `ChatOpenAI` 파라미터 전체 | [langchain-openai 레퍼런스](https://reference.langchain.com/python/langchain-openai) |

> **두 사이트의 역할이 다릅니다.** `docs.langchain.com` 은 **개념과 사용법**을 설명하는 안내서이고, `reference.langchain.com` 은 **클래스·인자 목록**을 그대로 보여 주는 사전입니다. "이게 왜 필요한가"는 앞쪽, "이 함수에 어떤 인자가 있나"는 뒤쪽에서 찾으면 빠릅니다.

**오늘의 목표**

- [ ] **왜 LangChain 인가** — 모델을 만드는 줄만 갈아 끼우면 되는 부품 표준화를 이해한다.
- [ ] **메시지와 모델** — `HumanMessage`·`SystemMessage`, `invoke`, 응답의 `.text` 를 익힌다.
- [ ] **이미지 입력** — 메시지 내용을 블록 리스트로 만들어 사진과 질문을 함께 보낸다.
- [ ] **프롬프트 템플릿** — `ChatPromptTemplate` 로 변수를 끼운 재사용 프롬프트를 만든다.
- [ ] **프롬프트를 파일로** — `prompts.yml` 을 읽어 템플릿을 만든다(코드 밖에서 문구 관리).
- [ ] **출력 파서** — `StrOutputParser` 로 응답에서 문자열만 뽑는다.
- [ ] **체인(LCEL)** — 파이프 `|` 로 `프롬프트 | 모델 | 파서` 를 잇고 `invoke`·`batch` 한다.
- [ ] **체인 잇기** — 앞 체인의 출력을 뒤 체인의 입력으로 넘겨 2단 흐름을 만든다.

아래 준비 셀을 먼저 실행하세요. **14일차에 쓰던 본인 OpenAI API 키가 필요합니다** — 일차 폴더에서 `.env.example` 을 `.env` 로 복사하고 `OPENAI_API_KEY` 를 채우면 됩니다. 오늘 오가는 요청은 짧은 문장 수십 건이라 비용은 아주 적습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

---
# 1. 왜 LangChain 인가 — 부품을 표준화한다

## 왜 필요할까요?
언어 모델을 만드는 회사는 여럿입니다(예: 구글·오픈AI). 회사마다 사용법이 조금씩 다른데, 실무에서는 **모델을 바꿔 가며 비교**하거나 **비용·성능 때문에 교체**하는 일이 잦습니다. 그때마다 코드를 고치면 곤란합니다.

## 비유
- **직접 호출** = 회사마다 다른 충전 단자. 기기를 바꾸면 케이블도 바꿔야 합니다.
- **LangChain** = 공통 규격 단자. 어느 회사 모델을 꽂아도 **쓰는 방법이 같습니다**.

## 문법 — `ChatOpenAI(model='gpt-4o-mini')`
LangChain 에서 모델은 **부품 하나**입니다. 회사마다 전용 부품이 있고, 우리는 오픈AI 부품인 **`ChatOpenAI`** 를 씁니다. `temperature` 같은 파라미터는 14일차에 배운 것과 같은 뜻입니다 — 0에 가까울수록 답이 일정합니다.

> `temperature=0` 이라도 **문장이 매번 똑같이 나오지는 않습니다.** 결이 비슷할 뿐이라, 오늘 실행 결과가 교재와 글자까지 같지 않아도 틀린 것이 아닙니다.

<img src="images/why_langchain.png" width="760">

In [ ]:
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('모델 부품 종류:', type(model).__name__)

# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

> **부품을 바꾸면 회사가 바뀝니다.** 구글 모델을 쓰려면 위 **두 줄만** `from langchain_google_genai import ChatGoogleGenerativeAI` / `model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')` 로 갈아 끼우면 됩니다. 아래에서 배울 **메시지·프롬프트·파서·체인 코드는 한 줄도 바꾸지 않습니다** — 이것이 부품 표준화의 힘입니다. (오늘은 오픈AI 키로 진행하니 `ChatOpenAI` 그대로 씁니다.)

### ✅ 바로 확인 퀴즈

**1.** LangChain 을 쓰면 언어 모델 회사를 바꿀 때 무엇이 편해지나요?

<details><summary>정답 보기</summary>

**사용 방법이 통일**되어, 모델을 바꿔도 `invoke`·프롬프트·파서·체인 코드를 그대로 쓸 수 있습니다. 바뀌는 것은 **모델 부품을 만드는 줄**뿐입니다.

</details>

**2.** 14일차의 `client.chat.completions.create(...)` 와 오늘의 모델 부품은 무엇이 다른가요?

<details><summary>정답 보기</summary>

14일차는 **오픈AI 전용 함수**를 직접 불렀습니다. 오늘의 모델 부품은 **어느 회사든 `invoke` 로 부르는 공통 규약**을 따르고, 그래서 프롬프트·파서와 파이프 `|` 로 이어 붙일 수 있습니다.

</details>

---
# 2. 메시지와 모델 — 모델에게 말 걸기

## 왜 이렇게 할까요?
챗 모델은 **대화**로 동작합니다. 그래서 입력을 **메시지** 단위로 줍니다. 가장 많이 쓰는 두 가지는:
- **`HumanMessage`**: 사용자가 하는 말(질문·요청).
- **`SystemMessage`**: 모델의 **역할·태도**를 정하는 말(예: "너는 친절한 상담원이다").

## 문법 — `invoke` 와 `.text`
모델에게 물어보려면 `model.invoke(...)` 를 씁니다. 문자열 하나를 넣으면 사용자의 말로 취급합니다. 돌아온 답은 **`AIMessage`** 라는 부품인데, **순수 텍스트**만 필요하면 **`.text`** 로 꺼냅니다.

> 응답에는 `.content` 도 있지만, 모델·설정에 따라 그 안이 **여러 조각의 리스트**일 수 있습니다. 사람이 읽을 **텍스트**는 어느 경우에나 `.text` 로 꺼낼 수 있어 안전합니다(이 관례를 오늘부터 계속 씁니다).

In [ ]:
# 가장 간단한 호출 — 문자열 하나를 넣으면 모델이 그것을 '사용자의 말'로 받아들입니다.
reply = model.invoke('강아지 방석을 처음 고르는 사람에게 확인할 점 두 가지를 짧게 알려줘.')

# 확인 포인트 1: 돌아온 것은 문자열이 아니라 AIMessage 라는 '부품'입니다.
# 확인 포인트 2: 사람이 읽을 텍스트는 그 부품에서 .text 로 꺼냅니다.
print('응답 부품 종류:', type(reply).__name__)
print('응답 텍스트  :', reply.text)

이번엔 **역할**을 정해 봅니다. `SystemMessage` 로 상담원 역할을 주고, `HumanMessage` 로 질문합니다. 메시지 여러 개는 **리스트**로 넣습니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# 메시지 여러 개는 리스트로 넣습니다. 순서가 중요합니다 —
#   역할을 정하는 SystemMessage 가 먼저, 사용자의 질문인 HumanMessage 가 뒤에 옵니다.
messages = [
    SystemMessage('너는 반려동물 용품 온라인숍의 친절한 상담원이다. 존댓말로 간결하게 답한다.'),
    HumanMessage('요즘 자꾸 밥을 남기는데 어떻게 하죠?'),
]
reply2 = model.invoke(messages)

# 확인 포인트: 질문은 그대로인데 역할을 준 것만으로 말투와 관점이 달라졌는지 읽어 보세요.
print(reply2.text)

### 🖐️ 함께 따라하기 — 역할을 바꿔 물어보기

위와 같은 질문을 하되, `SystemMessage` 의 역할을 **"수의학 지식이 있는 전문 상담원"** 으로 바꿔 `model.invoke([...])` 로 답을 받아 `.text` 로 출력해 보세요. 역할에 따라 답의 결이 어떻게 달라지는지 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) SystemMessage 역할을 '수의학 지식이 있는 전문 상담원' 으로 바꾼다
# 2) 같은 질문을 HumanMessage 로 넣어 model.invoke([...]) 를 호출한다
# 3) 응답의 .text 를 출력한다

## 이어서 — 이미지도 메시지에 담기

메시지에 담을 수 있는 것은 글자만이 아닙니다. **사진도 같은 메시지 안에** 넣을 수 있습니다. 상품 사진을 보여 주고 홍보 문구를 받거나, 매물 사진을 보고 상태를 설명하게 하는 식입니다.

14일차에서 이미 해 본 일입니다. **달라지는 것은 껍데기뿐**입니다 — 그때는 `client.chat.completions.create(messages=[...])` 였고, 오늘은 `model.invoke([HumanMessage(...)])` 입니다. **메시지 안에 넣는 모양은 글자 하나 바뀌지 않습니다.**

## 문법 — 내용(content)을 리스트로
지금까지 `HumanMessage('질문')` 처럼 **문자열 하나**를 넣었습니다. 사진을 함께 보내려면 내용을 **블록의 리스트**로 줍니다.

- 글자 블록: `{'type': 'text', 'text': '...'}`
- 사진 블록: `{'type': 'image_url', 'image_url': {'url': 사진주소}}`

사진 주소 자리에는 **웹 주소(https://...)** 를 넣을 수도 있고, **내 컴퓨터의 파일**을 `data URL` 이라는 긴 문자열로 바꿔 넣을 수도 있습니다. 여기서는 파일을 씁니다.

In [ ]:
import base64

def to_data_url(path):
    """로컬 이미지 파일을 모델에 넣을 수 있는 data URL 문자열로 바꾼다."""
    # 사진은 글자가 아니라서 그대로 못 보낸다 — base64 로 '글자처럼' 바꿔 실어 보낸다(14일차와 같은 방법).
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'data:image/jpeg;base64,{b64}'

def to_base64(path):
    """로컬 이미지 파일을 모델에 넣을 수 있는 base64 디코딩 데이터로 반환"""
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return b64

bed_url = to_data_url('data/images/pet_bed.jpg')
bed_b64 = to_base64('data/images/pet_bed.jpg')

# 확인 포인트: 사진 한 장이 아주 긴 '문자열' 하나가 되었다는 것.
print('앞부분:', bed_url[:40], '...')
print('전체 길이:', len(bed_url), '자')

모델에게 보여 줄 사진입니다(`data/images/pet_bed.jpg`).

<img src="data/images/pet_bed.jpg" width="360">

In [ ]:
# 내용(content)을 블록 리스트로 — 글자 블록 하나 + 사진 블록 하나.
#   SystemMessage 로 역할을 주는 방법은 위와 똑같다(메시지 종류는 그대로, 내용의 모양만 바뀐 것).
photo_reply = model.invoke([
    SystemMessage('너는 반려동물 용품 홍보 문구를 쓰는 카피라이터다.'),
    HumanMessage(content=[
        {'type': 'text', 'text': '이 상품 사진을 보고 홍보 문구를 한 문장으로 써줘. 사진에서 보이는 특징을 근거로 들어줘.'},
        {'type': 'image_url', 'image_url': {'url': bed_url}},
    ]),
])

# 확인 포인트: 답이 '사진에서 실제로 보이는 것'(색·두께·지퍼 등)을 근거로 삼았는지 읽어 보세요.
print(photo_reply.text)

In [ ]:
# 내용(content)을 블록 리스트로 — 글자 블록 하나 + 사진 블록 하나.
#   SystemMessage 로 역할을 주는 방법은 위와 똑같다(메시지 종류는 그대로, 내용의 모양만 바뀐 것).
photo_reply = model.invoke([
    SystemMessage('너는 반려동물 용품 홍보 문구를 쓰는 카피라이터다.'),
    HumanMessage('이 상품 사진을 보고 홍보 문구를 한 문장으로 써줘. 사진에서 보이는 특징을 근거로 들어줘.'),
    HumanMessage(content=[
        # {'type': 'image_url', 'image_url': {'url': bed_url}}, # <-- OpenAI에 특화된 방법
        
        # 모든 LLM 벤더에 상관없이 이미지를 넣는 langchain의 표준화된 방법
        {'type': 'image', 'source_type': 'base64', 'data': bed_b64, 'mime_type': 'image/jpeg'}
    ]),
])

# 확인 포인트: 답이 '사진에서 실제로 보이는 것'(색·두께·지퍼 등)을 근거로 삼았는지 읽어 보세요.
print(photo_reply.text)

mime_type은 base64 문자열이 **무슨 형식의 파일인지* 알려주는 라벨

| 확장자 | mime_type |
|---|---|
| `.jpg`, `.jpeg` | `image/jpeg` |
| `.png` | `image/png` |
| `.gif` | `image/gif` |
| `.webp` | `image/webp` |
| `.pdf` | `application/pdf` |

> 답이 `AIMessage` 로 돌아오고 `.text` 로 꺼내는 것까지 **글자만 보낼 때와 완전히 같습니다.** 바뀐 것은 **보낸 내용의 모양** 하나뿐입니다.

**주의 두 가지.** 사진은 글자보다 **훨씬 비쌉니다**(큰 사진일수록 더). 그래서 실무에서는 보내기 전에 크기를 줄입니다 — 이 교재의 사진도 가로 768픽셀로 줄여 둔 것입니다. 그리고 모델은 **사진에 없는 것도 그럴듯하게 지어냅니다.** 답을 그대로 상품 페이지에 쓰지 말고 사진과 대조해 보세요.

### 🖐️ 함께 따라하기 — 사진으로 상품 정보 뽑기

같은 사진(`bed_url`)을 쓰되 **요청을 바꿔** 보세요. 홍보 문구 대신 **"이 상품의 색깔과 형태를 각각 한 단어로만 알려줘"** 라고 물어 답을 출력하세요.

**확인 기준**: 사진에 실제로 보이는 색(회색 계열)과 형태(원형)가 답에 들어온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) HumanMessage 의 content 를 [글자 블록, 사진 블록] 리스트로 만든다
# 2) 글자 블록의 질문만 '색깔과 형태를 각각 한 단어로만' 으로 바꾼다
# 3) model.invoke 로 보내고 .text 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 모델의 **역할·태도**를 정하는 메시지는 무엇인가요?

<details><summary>정답 보기</summary>

**`SystemMessage`** 입니다. 사용자의 질문은 `HumanMessage` 로 넣습니다.

</details>

**2.** 응답에서 순수 텍스트를 안전하게 꺼내려면 무엇을 쓰나요?

<details><summary>정답 보기</summary>

`응답.text` 를 씁니다. `.content` 는 여러 조각의 리스트일 수 있어 `.text` 가 안전합니다.

</details>

**3.** 사진을 함께 보내려면 `HumanMessage` 의 무엇을 바꿔야 하나요?

<details><summary>정답 보기</summary>

**내용(content)** 을 문자열 하나 대신 **블록의 리스트**로 줍니다 — 글자 블록(`{'type': 'text', ...}`)과 사진 블록(`{'type': 'image_url', ...}`)을 함께 넣습니다. 메시지 종류(`HumanMessage`)와 답을 꺼내는 방법(`.text`)은 그대로입니다.

</details>

---
# 3. 프롬프트 템플릿 — 변수를 끼운 재사용 프롬프트

## 왜 필요할까요?
실무에서는 **같은 형식의 프롬프트를 값만 바꿔** 수백 번 씁니다(상품마다 홍보 문구, 고객마다 답변). 매번 문자열을 이어 붙이면 실수가 잦습니다. **프롬프트 템플릿**은 `{변수}` 자리를 비워 두고, 나중에 값을 채워 완성된 프롬프트를 만들어 줍니다.

## 문법 — `ChatPromptTemplate.from_messages`
- `ChatPromptTemplate.from_messages([...])` 에 (역할, 내용) 쌍의 리스트를 줍니다.
- 내용 안의 **`{변수}`** 는 나중에 `invoke({'변수': 값})` 로 채웁니다.
- 완성된 프롬프트만 눈으로 보고 싶으면 `.invoke(...)` 대신 `.format_messages(...)` 로 확인할 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 중괄호 {name}·{keywords} 가 나중에 값으로 채워질 '빈칸'입니다.
#   메시지를 (역할, 내용) 튜플로 적으면 SystemMessage/HumanMessage 를 직접 만들지 않아도 됩니다.
promo_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 홍보 문구를 쓰는 카피라이터다.'),
    ('human', '다음 상품의 홍보 문구를 한 문장으로 써줘. 이름: {name}, 특징: {keywords}'),
])

# 모델을 부르기 전에, 빈칸이 채워진 프롬프트가 어떻게 생겼는지 먼저 눈으로 봅니다.
#   (format_messages 는 프롬프트만 완성할 뿐 모델을 호출하지 않습니다 — 요금도 들지 않습니다.)
filled = promo_prompt.format_messages(name='포근 강아지 방석', keywords='메모리폼, 미끄럼 방지 바닥, 세탁 가능')
print(filled)

# 확인 포인트: 메시지 두 개가 만들어졌고, {name}·{keywords} 자리에 값이 들어갔는지.
for m in filled:
    print(type(m).__name__, ':', m.content)

이제 완성된 프롬프트를 모델에 넣어 봅니다. 템플릿도 `invoke` 로 값을 채우고, 그 결과를 모델에 넘깁니다.

In [ ]:
# 템플릿의 invoke 는 '완성된 프롬프트'를 만들 뿐 모델을 부르지 않는다 — 부르는 것은 다음 줄.
filled_prompt = promo_prompt.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'})
print(filled_prompt)

print(model.invoke(filled_prompt).text)

> **보조 표기 — 한 줄짜리 간단한 템플릿은 `from_template`**
역할(system/human)을 나눌 필요 없이 **사람 메시지 하나**만 필요하면, `ChatPromptTemplate.from_template('... {변수} ...')` 로 더 짧게 만들 수 있습니다. 채우는 방법(`invoke({'변수': 값})`)은 똑같습니다. 다음 단원 복습에서도 이 표기가 나옵니다.

In [ ]:
# 역할 구분이 필요 없을 땐 from_template 한 줄로 프롬프트를 만들 수 있습니다.
#   변수 뒤에 조사('를'·'을')를 바로 붙이면 값에 따라 어색해진다 — 쉼표로 끊어 두면 어떤 상품이 와도 자연스럽다.
one_line_prompt = ChatPromptTemplate.from_template('{name}, 한 문장으로 홍보해줘.')
filled_one = one_line_prompt.invoke({'name': '포근 강아지 방석'})
print(filled_one)
print(filled_one.messages[0].content)   # 완성된 사람 메시지 내용

### 🖐️ 함께 따라하기 — 다른 상품으로 문구 만들기

같은 `promo_prompt` 를 그대로 쓰되, 상품을 **이름="튼튼 고양이 스크래처", 특징="고밀도 골판지, 캣닢 포함, 교체형"** 로 바꿔 완성된 프롬프트를 모델에 넣고 답을 `.text` 로 출력해 보세요. 템플릿 하나로 상품만 갈아 끼우는 감을 잡습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) promo_prompt.invoke 에 새 상품(name·keywords)을 넣어 프롬프트를 완성한다
# 2) 완성된 프롬프트를 model.invoke 에 넣는다
# 3) 응답의 .text 를 출력한다

## 이어서 — 프롬프트를 파일로 관리하기

지금까지 프롬프트를 **코드 안에** 적었습니다. 그런데 실무에서 가장 자주 바뀌는 것이 바로 프롬프트입니다. 문구 한 줄 고치자고 **코드를 고치고 다시 배포**하는 것은 번거롭고, 문구를 다듬을 사람(기획·운영)이 **파이썬을 몰라도** 고칠 수 있어야 합니다.

그래서 프롬프트를 **코드가 아니라 자료**로 다룹니다 — 파일에 두고 읽어 씁니다.

## 왜 YAML 인가
프롬프트는 **여러 줄**이고 따옴표·중괄호가 섞입니다. JSON 에 넣으면 줄바꿈을 전부 `\n` 으로 바꿔야 해서 **사람이 읽기 어려워집니다.** YAML 은 `|` 뒤에 여러 줄을 그대로 쓸 수 있어, **파일에 보이는 글자가 곧 프롬프트**입니다.

아래가 이 단원이 쓰는 프롬프트 파일 `data/prompts.yml` 입니다.

```yaml
promo_writer:
  description: 상품 이름과 특징을 받아 홍보 문구 한 문장을 쓴다
  system: |
    너는 반려동물 용품 홍보 문구를 쓰는 카피라이터다.
    과장하지 않고, 주어진 특징에 근거해서만 쓴다.
  human: |
    다음 상품의 홍보 문구를 한 문장으로 써줘.
    이름: {name}
    특징: {keywords}
```

## 문법 — `yaml.safe_load` 로 읽어 템플릿으로
파일을 읽으면 그냥 **딕셔너리**입니다. 그 안의 `system`·`human` 글을 `ChatPromptTemplate.from_messages` 에 넣어 주면 **코드에 적었을 때와 똑같은 부품**이 됩니다.

In [ ]:
import yaml

# safe_load 는 YAML 을 파이썬 딕셔너리로 바꿔 준다(load 가 아니라 safe_load 를 쓴다 — 임의 코드 실행을 막는다).
with open('data/prompts.yml', encoding='utf-8') as f:
    PROMPTS = yaml.safe_load(f)

# 확인 포인트: 파일 하나에 프롬프트 여러 개를 이름표로 담아 둔다.
print('파일에 든 프롬프트:', list(PROMPTS))
print('promo_writer 설명:', PROMPTS['promo_writer']['description'])

from pprint import pprint
pprint(PROMPTS)

이제 그 글을 템플릿 부품으로 만듭니다. **읽어 온 문자열을 넣을 뿐**, 만드는 방법은 앞과 똑같습니다.

In [ ]:
def load_prompt(name):
    """프롬프트 파일에서 이름으로 꺼내 ChatPromptTemplate 로 만든다."""
    # 파일에서 온 글이든 코드에 적은 글이든, from_messages 입장에서는 그냥 문자열이다.
    spec = PROMPTS[name]
    return ChatPromptTemplate.from_messages([
        ('system', spec['system']),
        ('human', spec['human']),
    ])

file_prompt = load_prompt('promo_writer')
print(file_prompt)

# 확인 포인트: 파일에서 왔어도 {name}·{keywords} 를 그대로 알아본다 — 코드에 적었을 때와 같은 부품이다.
print('이 프롬프트가 요구하는 변수:', sorted(file_prompt.input_variables))

In [ ]:
# 채우고 부르는 방법도 앞과 똑같다 — 바뀐 것은 '프롬프트가 어디에 적혀 있는가' 뿐이다.
filled_file = file_prompt.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'})
print(model.invoke(filled_file).text)

> **코드는 한 줄도 안 바꾸고** 문구만 바꾸고 싶으면 이제 `prompts.yml` 만 열어 고치면 됩니다. 프롬프트가 **코드에서 자료로** 옮겨 간 것입니다.

실무에서는 여기에 **버전**을 더해 `v1`·`v2` 를 나란히 두고 어느 쪽이 더 좋은지 비교하기도 합니다. 그 이야기는 **뒤 단원(에이전트 품질·관측성)** 에서 다룹니다 — 오늘은 "프롬프트를 파일에서 읽어 쓴다" 까지입니다.

### 🖐️ 함께 따라하기 — 파일의 다른 프롬프트 쓰기

같은 파일에는 **`care_guide`** 라는 프롬프트도 들어 있습니다. `load_prompt('care_guide')` 로 꺼내 **이름="튼튼 고양이 스크래처"** 을 채워 모델에 넣고 답을 출력해 보세요.

**확인 기준**: 그 프롬프트가 요구하는 변수는 `name` 하나다(`sorted(...input_variables)` 로 확인).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) load_prompt('care_guide') 로 프롬프트를 꺼낸다
# 2) input_variables 를 출력해 어떤 변수를 요구하는지 확인한다
# 3) name 에 '튼튼 고양이 스크래처' 을 채워 model.invoke 에 넣고 .text 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 템플릿의 `{name}` 자리에 값을 채우려면 어떻게 하나요?

<details><summary>정답 보기</summary>

`프롬프트.invoke({'name': 값, ...})` 처럼 **딕셔너리**로 변수 값을 넘깁니다.

</details>

**2.** 프롬프트 템플릿을 쓰는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

**같은 형식의 프롬프트를 값만 바꿔 재사용**하기 위해서입니다. 문자열을 매번 이어 붙이는 실수를 줄여 줍니다.

</details>

**3.** 프롬프트를 코드가 아니라 파일에 두면 무엇이 좋아지나요?

<details><summary>정답 보기</summary>

**코드를 고치지 않고 문구만 바꿀 수 있습니다.** 파이썬을 모르는 사람도 파일만 열어 고칠 수 있고, 코드 재배포 없이 프롬프트를 교체할 수 있습니다. 여러 줄 글이라 `\n` 이스케이프가 없는 **YAML** 이 읽기 편합니다.

</details>

---
# 4. 출력 파서 — 응답에서 필요한 형태만 뽑기

## 왜 필요할까요?
모델의 답은 `AIMessage` 라는 부품입니다. 하지만 우리가 뒤에서 쓰고 싶은 건 대개 **그냥 문자열**입니다. 매번 `.text` 를 부르는 대신, **출력 파서**를 체인 끝에 두면 자동으로 문자열만 뽑아 줍니다.

## 문법 — `StrOutputParser`
`StrOutputParser()` 는 `AIMessage` 를 받아 **문자열**로 바꿔 주는 부품입니다. 다음 절에서 이 파서를 체인 맨 끝에 붙입니다.

> **출력 파서는 한 종류가 아닙니다.** 파서는 "모델의 답을 **다음 단계가 쓰기 좋은 모양**으로 바꾸는 부품" 을 통틀어 부르는 이름이고, 그중 가장 기본이 문자열로 바꾸는 `StrOutputParser` 입니다. 답을 **표·JSON 처럼 정해진 틀**로 받아 프로그램에 바로 꽂는 방법은 다음 단원에서 따로 배웁니다 — 오늘은 **문자열 하나**로 충분한 흐름만 다룹니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# 파서도 '부품' 하나입니다 — 모델처럼 invoke 로 씁니다.
parser = StrOutputParser()

# 먼저 모델을 불러 응답 부품(AIMessage)을 받습니다.
ai_message = model.invoke('포근 강아지 방석, 한마디로 홍보해줘.')

# 그 부품을 파서에 통과시키면 순수 문자열만 남습니다.
text_only = parser.invoke(ai_message)

# 확인 포인트: 종류가 AIMessage 에서 str 로 바뀌었다는 것.
#   내용은 같습니다 — 파서가 하는 일은 '껍데기를 벗기는 것'뿐입니다.
print('파서 통과 전 종류:', type(ai_message).__name__)
print('파서 통과 후 종류:', type(text_only).__name__)
print(text_only)

### 🖐️ 함께 따라하기 — 다른 응답도 문자열로

다른 상품에 대한 응답을 파서로 바꿔 보세요. `model.invoke('산책 자동 리드줄, 한마디로 홍보해줘.')` 결과(AIMessage)를 `parser.invoke(...)` 에 넣어 순수 문자열로 만든 뒤 출력하고, 종류가 `str` 인지 확인하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) model.invoke('산책 자동 리드줄, 한마디로 홍보해줘.') 로 응답을 받는다
# 2) parser.invoke(응답) 으로 문자열을 만든다
# 3) 종류(type)와 내용을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `StrOutputParser` 는 무엇을 무엇으로 바꿔 주나요?

<details><summary>정답 보기</summary>

모델 응답인 **`AIMessage` 를 순수 문자열(`str`)** 로 바꿔 줍니다.

</details>

---
# 5. 체인(LCEL) — 파이프 `|` 로 부품 잇기

## 왜 이렇게 할까요?
지금까지 **프롬프트 → 모델 → 파서** 를 따로따로 불렀습니다. 이 셋은 거의 항상 **이 순서**로 함께 쓰이므로, LangChain 은 이들을 **파이프 `|`** 로 이어 하나의 **체인**으로 묶게 해 줍니다. 이 표기법을 **LCEL** (LangChain Expression Language)이라고 부릅니다.

## 문법 — `prompt | model | parser`
- `chain = 프롬프트 | 모델 | 파서` 로 부품을 왼쪽에서 오른쪽으로 잇습니다.
- `chain.invoke({변수})` 하나면 프롬프트 채우기 → 모델 호출 → 문자열 뽑기가 **한 번에** 됩니다.
- 입력이 여러 개면 `chain.batch([{...}, {...}])` 로 **한꺼번에** 처리합니다.

> 파이프 `|` 는 파이썬이 원래 갖고 있던 기호인데, LangChain 이 **부품끼리 만났을 때의 뜻을 "이어 붙이기"로 정해 둔** 것입니다. 그래서 새 문법을 배우는 것이 아니라, **이 기호를 이 라이브러리가 어떻게 읽는지**만 알면 됩니다. 이어 붙인 결과도 다시 **부품 하나**라, 그 자체를 또 다른 체인에 넣을 수 있습니다.

<img src="images/lcel_chain.png" width="760">

In [ ]:
# 앞에서 따로따로 쓰던 세 부품을 파이프로 잇습니다. 왼쪽 출력이 오른쪽 입력으로 흘러갑니다.
#   promo_prompt(빈칸 채우기) -> model(답 생성) -> parser(문자열만 남기기)
promo_chain = promo_prompt | model | parser

# 이제 invoke 한 번이면 세 단계가 모두 끝납니다.
one = promo_chain.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'})

# 확인 포인트: 파서가 맨 끝에 있으므로 결과가 AIMessage 가 아니라 곧바로 str 입니다.
#   앞 절처럼 .text 를 따로 부르지 않아도 됩니다.
print('결과의 종류:', type(one).__name__)
print(one)

여러 상품의 문구를 **한꺼번에** 만들어 봅니다. 상품 목록은 손으로 적는 대신 **실제 데이터 파일**(`data/pet_products.csv`)에서 읽어 옵니다(지난 단원 `read_csv` 복습). 각 행의 `name`·`keywords` 열로 입력 딕셔너리들의 리스트를 만들어 `batch` 에 넘깁니다.

In [ ]:
import pandas as pd

# (1) 상품 데이터를 파일에서 읽습니다 — 어떤 열이 있는지부터 눈으로 확인하세요.
products_df = pd.read_csv('data/pet_products.csv')
print('전체 상품 수:', len(products_df))
display(products_df.head(3))

표의 `name`·`keywords` 열을 프롬프트 변수 이름에 맞춰 **딕셔너리들의 리스트**로 바꿉니다. 이 모양이 바로 `batch` 가 받는 입력입니다 — 모델을 부르기 전에 **모양부터** 확인합니다.

In [ ]:
# (2) 각 행을 {'name': ..., 'keywords': ...} 딕셔너리로 — 프롬프트의 변수 이름과 같아야 합니다.
products = [
    {'name': row['name'], 'keywords': row['keywords']}
    for _, row in products_df.head(3).iterrows()
]

# 확인 포인트: 아직 모델을 부르지 않았습니다. 입력 모양만 만들어 본 것입니다.
print('batch 에 넣을 개수:', len(products))
for p in products:
    print(' ', p)

이제 만든 리스트를 `batch` 에 넘깁니다.

In [ ]:
products

In [ ]:
# (3) batch: 입력 리스트 -> 결과 리스트. 넣은 순서 그대로 돌아옵니다.
results = promo_chain.batch(products)

# 확인 포인트: 넣은 개수와 받은 개수가 같고, 상품과 문구가 짝이 맞는지.
print('넣은 개수:', len(products), '/ 받은 개수:', len(results))
for p, r in zip(products, results):
    print('-', p['name'], '→', r)

> `batch` 는 세 요청을 **동시에** 보내므로 하나씩 `invoke` 하는 것보다 빠릅니다. 결과는 그래도 **입력 순서 그대로** 돌아옵니다. 한 번에 보내는 개수를 줄이고 싶으면(예: 요금제의 분당 요청 한도) `chain.batch(입력들, config={'max_concurrency': 1})` 처럼 제한할 수 있습니다.

### 🖐️ 함께 따라하기 — 나만의 3단 체인 만들기

이번엔 처음부터 만들어 봅니다. **상품 특징을 받아 한 줄 상품평을 쓰는** 체인을 조립하세요.

1. `ChatPromptTemplate.from_messages` 로 `{name}` 특징을 받는 프롬프트를 만든다(역할·요청 자유).
2. 그 프롬프트에 `model`, `parser` 를 파이프 `|` 로 이어 `review_chain` 을 만든다.
3. `review_chain.invoke({'name': '무소음 급수기'})` 를 출력한다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) {name} 상품의 한 줄 상품평을 요청하는 프롬프트 템플릿을 만든다
# 2) 프롬프트 | model | parser 로 review_chain 을 만든다
# 3) review_chain.invoke({'name': '무소음 급수기'}) 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `프롬프트 | 모델 | 파서` 에서 파이프 `|` 는 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

왼쪽 부품의 **출력을 오른쪽 부품의 입력으로** 넘겨 잇는다는 뜻입니다. 프롬프트→모델→파서 순으로 흐릅니다.

</details>

**2.** 입력 여러 개를 한꺼번에 처리하는 메서드는?

<details><summary>정답 보기</summary>

**`batch([...])`** 입니다. 입력 딕셔너리들의 리스트를 넘기면 결과 리스트를 돌려줍니다.

</details>

---
# 6. 체인 잇기 — 앞 결과를 뒤로 넘기기

## 왜 이렇게 할까요?
복잡한 작업은 **여러 단계**로 나누면 품질이 좋아집니다. 예를 들어 (1) 먼저 상세한 홍보 문구를 쓰고, (2) 그것을 **짧은 한 줄 광고**로 다듬는 식입니다. 앞 체인의 **문자열 출력**을 뒤 체인의 **입력**으로 넘기면 이런 2단 흐름이 됩니다.

## 문법 — 출력을 다음 프롬프트 변수로
앞 체인이 문자열을 내므로, 뒤 프롬프트가 그 문자열을 받을 **변수 하나**(`{draft}`)를 두면 됩니다. 두 체인을 각각 만들고, 앞 결과를 뒤 `invoke` 의 값으로 넣습니다.

**(1단계) 먼저 상세한 홍보 문구를 만듭니다.** 앞 절에서 만든 `promo_chain` 을 그대로 재사용합니다.

In [ ]:
# 1단계 결과를 draft 라는 변수에 담아 둡니다 — 이것을 2단계의 입력으로 쓸 것입니다.
draft = promo_chain.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'})
print('[1단계 초안]', draft)

**(2단계) 초안을 짧게 다듬을 두 번째 체인을 만듭니다.** 이 프롬프트는 앞 결과를 받을 변수 `{draft}` 하나만 둡니다 — 아직 실행하지는 않습니다.

In [ ]:
# 받을 변수 이름이 {draft} 라는 점이 중요합니다 — 아래에서 이 이름으로 값을 넘깁니다.
shorten_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 긴 문구를 짧고 강한 한 줄 광고로 다듬는 편집자다.'),
    ('human', '다음 문구를 12자 안팎의 한 줄 광고로 줄여줘:\n{draft}'),
])
shorten_chain = shorten_prompt | model | parser
print('2단계 체인 준비 완료')

**(3단계) 1단계 결과를 2단계에 넘깁니다.** 두 번의 `invoke` 를 손으로 이어 준 셈입니다 — 다음 시간에는 이 두 체인을 **하나로 잇는** 방법을 배웁니다.

In [ ]:
# 앞 결과(draft)를 뒤 프롬프트의 {draft} 자리에 넣어 줍니다.
final = shorten_chain.invoke({'draft': draft})

# 확인 포인트: 같은 상품인데 1단계는 길고, 2단계는 짧아졌는지 비교해 보세요.
print('[1단계 초안 ]', draft)
print('[2단계 한 줄]', final)

> 앞 체인의 **문자열 출력**을 뒤 체인의 `{draft}` 변수로 그대로 넘겼습니다. 이렇게 단계를 나누면 각 단계가 **한 가지 일**만 해서 결과를 다루기 쉽습니다. (다음 단원에서는 이 연결을 더 매끄럽게 해 주는 `Runnable` 부품들을 배웁니다.)

### 🖐️ 함께 따라하기 — 다른 상품으로 2단 흐름

같은 두 체인을 다른 상품에 이어 보세요. `promo_chain` 으로 **이름="튼튼 고양이 스크래처", 특징="고밀도 골판지, 캣닢 포함, 교체형"** 초안을 만든 뒤, 그 초안을 `shorten_chain` 에 넣어 한 줄 광고로 줄이고, 두 결과를 모두 출력하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) promo_chain 으로 '튼튼 고양이 스크래처'(특징 '고밀도 골판지, 캣닢 포함, 교체형') 초안을 만든다
# 2) 그 초안을 shorten_chain 에 넣어 한 줄로 줄인다
# 3) 초안과 한 줄 결과를 모두 출력한다

### ✅ 바로 확인 퀴즈

**1.** 2단 체인에서 앞 체인의 출력은 뒤 체인에 어떻게 전달되나요?

<details><summary>정답 보기</summary>

앞 체인의 **문자열 결과**를 뒤 프롬프트의 **변수 값**(예: `{'draft': draft}`)으로 넣어 전달합니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| 모델 부품 | 공급자 교체가 쉬운 표준 모델 | `ChatOpenAI(model='gpt-4o-mini')` |
| 메시지 | 역할·질문을 메시지로 | `SystemMessage`·`HumanMessage`, `.invoke`, `.text` |
| 이미지 입력 | 사진과 질문을 한 메시지에 | `HumanMessage(content=[글자 블록, 사진 블록])` |
| 프롬프트 템플릿 | 변수 끼운 재사용 프롬프트 | `ChatPromptTemplate.from_messages` |
| 프롬프트 파일 | 문구를 코드 밖 YAML 로 | `yaml.safe_load` → `from_messages` |
| 출력 파서 | 응답에서 문자열만 | `StrOutputParser` |
| 체인(LCEL) | 파이프로 부품 잇기 | `프롬프트 \| 모델 \| 파서`, `.invoke`·`.batch` |
| 체인 잇기 | 앞 출력 → 뒤 입력 | `chain2.invoke({'draft': draft})` |

- **부품을 표준화**하면 모델 교체·재사용·연결이 쉬워집니다.
- 거의 모든 흐름은 **프롬프트 → 모델 → 파서** 체인에서 출발합니다.

## ⏭️ 예고 — 다음 시간

다음 시간에는 모든 부품의 공통 규약인 **Runnable**(`invoke`·`batch`·**`stream`**)을 배우고, 내 파이썬 함수를 체인 부품으로 만드는 **RunnableLambda**, 입력을 나눠 **동시에 처리**하는 **RunnableParallel**, 그리고 모델이 앞 대화를 **기억**하게 하는 **대화 기록(Memory)** 워크플로우를 만듭니다.

수고하셨습니다!